# Chapter 4 notebook — Post-training quantization (PTQ)

Walks through `docs/08_model_optimization.md`:
1. Start from the FP32 ONNX model (Chapter 5).
2. Quantize to INT8 dynamic (weights only).
3. Quantize to INT8 static (weights + activations, calibration on random inputs).
4. Compare size, latency, and argmax agreement vs FP32.
5. Discuss why INT8 dynamic can be *slower* on small CNNs on CPU.

Run from the repo root.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import onnxruntime as ort
from src.optimization import (
    quantize_onnx_dynamic, quantize_onnx_static,
    RandomImageCalibrationDataReader, compare_models,
)
from src.optimization.quantization import argmax_agreement

OUT = Path('../experiments/exported_models')
SRC = OUT / 'mobilenet_v3_small.onnx'
DYN = OUT / 'mobilenet_v3_small_int8_dynamic.onnx'
STA = OUT / 'mobilenet_v3_small_int8_static.onnx'
assert SRC.exists(), f'missing {SRC}. Run Chapter 2 notebook first.'
print('onnxruntime', ort.__version__)
print('source FP32 model:', SRC, f'{SRC.stat().st_size / 1024 / 1024:.2f} MB')

## 1. Dynamic quantization (weights → INT8)

In [ ]:
quantize_onnx_dynamic(SRC, DYN)
print(f'wrote {DYN}  {DYN.stat().st_size / 1024 / 1024:.2f} MB')

## 2. Static quantization (weights + activations)

We use `RandomImageCalibrationDataReader` here because it requires no dataset. **For real workloads, replace it with a reader that yields real preprocessed validation images.** With random calibration, activation ranges will be off, so argmax agreement will be artificially low on real images.

In [ ]:
reader = RandomImageCalibrationDataReader('input', (1, 3, 224, 224), num_samples=64, seed=0)
quantize_onnx_static(SRC, STA, reader)
print(f'wrote {STA}  {STA.stat().st_size / 1024 / 1024:.2f} MB')

## 3. Compare on a synthetic batch

Argmax agreement vs FP32 is the simple accuracy proxy. With random inputs and random calibration it is mostly noise — but the size and latency numbers are honest.

In [ ]:
sample_inputs = [np.random.randn(1, 3, 224, 224).astype(np.float32) for _ in range(20)]
rows = compare_models(
    {'fp32': SRC, 'int8_dyn': DYN, 'int8_static': STA},
    sample_inputs=sample_inputs,
    reference_label='fp32',
    metric_fn=argmax_agreement,
    warmup=5, repeat=30,
)
print(f'{"label":<14} {"size MB":>8} {"P50 ms":>8} {"P95 ms":>8} {"FPS":>8} '
      f'{"max|diff|":>10} {"argmax agree":>13}')
for r in rows:
    print(f"{r['label']:<14} {r['size_mb']:>8.2f} {r['latency_p50_ms']:>8.2f} "
          f"{r['latency_p95_ms']:>8.2f} {r['fps_estimate']:>8.1f} "
          f"{r.get('max_abs_diff_vs_ref', float('nan')):>10.4f} "
          f"{r.get('metric_vs_ref', float('nan')):>13.3f}")

## 4. Argmax agreement on REAL ImageNet images (small held-out batch)

To get a meaningful accuracy proxy you need real images. The sample.jpg we have is synthetic; if you have access to ImageNet (or any image classification dataset), generate a batch of preprocessed inputs and rerun the comparison.

In [ ]:
from PIL import Image
from torchvision.models import MobileNet_V3_Small_Weights

preprocess = MobileNet_V3_Small_Weights.IMAGENET1K_V1.transforms()
sample_img = Image.open('../datasets/sample.jpg').convert('RGB')
real_x = preprocess(sample_img).unsqueeze(0).numpy().astype(np.float32)

rows_real = compare_models(
    {'fp32': SRC, 'int8_dyn': DYN, 'int8_static': STA},
    sample_inputs=[real_x],
    reference_label='fp32',
    metric_fn=argmax_agreement,
)
print(f'{"label":<14} {"size MB":>8} {"P50 ms":>8} {"max|diff|":>10} {"argmax agree":>13}')
for r in rows_real:
    print(f"{r['label']:<14} {r['size_mb']:>8.2f} {r['latency_p50_ms']:>8.2f} "
          f"{r.get('max_abs_diff_vs_ref', float('nan')):>10.4f} "
          f"{r.get('metric_vs_ref', float('nan')):>13.3f}")
print('Note: on a single image, argmax_agreement is binary (0 or 1).')
print('On a real validation set of >=100 images, expect >0.98 for INT8 static with proper calibration.')

## 5. Discussion

What you should observe (typical on a 16-core x86 CPU laptop):

1. **Size dropped ~4×** for both INT8 variants.
2. **INT8 dynamic is SLOWER than FP32 ORT** for this small CNN — per-call activation quantization overhead overwhelms the INT8 matmul savings.
3. **INT8 static is faster than dynamic** but still often slower than FP32 ORT on a beefy CPU.
4. **Where INT8 wins clearly:** NPU / TPU / Jetson INT8 paths, Raspberry Pi / ARM, large models. The course example illustrates that *size win is universal, speed win is hardware-dependent*.

Always benchmark on the target deployment device, not the dev machine.